In [744]:
import pandas as pd
import geopandas as gpd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import numpy as np
from plotly.subplots import make_subplots

In [745]:
#df_patients = pd.read_csv("Resultats/Excels/patients_FR_geocoded_adulte_clinique_patho.csv",sep=";")
df_patients_revenus_chomage = pd.read_csv('data/data_cleaned/patients_FR_revenus_chomage.csv', sep =';')

gdf_patients = gpd.GeoDataFrame(pd.read_csv('Resultats/Excels/patients_FR_geocoded_adulte_clinique_patho.csv',sep=";"), 
                       geometry=gpd.points_from_xy(pd.read_csv("Resultats/Excels/patients_FR_geocoded_adulte_clinique_patho.csv",sep=";")['x'], 
                                                   pd.read_csv("Resultats/Excels/patients_FR_geocoded_adulte_clinique_patho.csv",sep=";")['y']),
                       crs="EPSG:4326")

df_revenus = pd.read_csv("data/insee_revenu/BASE_TD_FILO_DISP_IRIS_2020.csv",sep=";",dtype=str)[["IRIS","DISP_MED20"]]

#import du shapefile des departements d'IDF
gdf_dept = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/departements/IDF/dpt_idf.shp')
len(gdf_patients)




C:\Users\lpokambo\AppData\Local\Temp\ipykernel_20512\1517814224.py:2: DtypeWarning:

Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.

C:\Users\lpokambo\AppData\Local\Temp\ipykernel_20512\1517814224.py:4: DtypeWarning:

Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.

C:\Users\lpokambo\AppData\Local\Temp\ipykernel_20512\1517814224.py:5: DtypeWarning:

Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.

C:\Users\lpokambo\AppData\Local\Temp\ipykernel_20512\1517814224.py:6: DtypeWarning:

Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.



57878

# DESCRIPTION DE LA COHORTE DES FEMMES

In [746]:
gdf_F = gdf_patients[gdf_patients["patient_sexe"]=="F"]

print(len(gdf_F))
print (f"{len(gdf_F)} des patients sur {len(gdf_patients)} sont des femmes soit {((len(gdf_F)/len(gdf_patients))*100).__round__(2)}% de la patientèle")

44877
44877 des patients sur 57878 sont des femmes soit 77.54% de la patientèle


In [747]:
gdf_F_patho = gdf_F.groupby("patho").size().reset_index(name='total')
gdf_F_patho

,patho,total
0,Autre,111
1,Dermato,535
2,Endocrino,696
3,Gastro,1040
4,Gynéco,2355
5,Hemato,1097
6,Neuro,354
7,ORL,446
8,Ophtalmo,2111
9,Sarcome,766


In [748]:
fig = px.bar(gdf_F_patho, x='patho', y='total', title='Nombre de patients par pathologie',
             labels={'patho': 'Pathologie', 'total': 'total patients'})
fig.show()

In [749]:
pathologie = ['Hemato','Sein','Gynéco','Thorax']

#### Repartition par type de cancer

In [750]:
#tab = pd.DataFrame(columns=["cancer","nb_patients","pourcentage"])
tab=[]
for patho in pathologie:
    df_patho = gdf_F[gdf_F['patho']== patho]
    tab.append({"cancer": patho , "nb_patients": len(df_patho), "pourcentage":(len(df_patho)/len(gdf_F)*100).__round__(2)})
tab = pd.DataFrame(tab)
sum(tab['nb_patients'])

38718

In [751]:
dep_idf = ["75","77","78","91","92","93","94","95"]
gdf_patientes_idf = gdf_F[gdf_F["INSEE_REG"]==11]
gdf_patientes_idf = gdf_patientes_idf[gdf_patientes_idf["CODE_DEPT"].isin(dep_idf)]

len(gdf_patientes_idf)


35818

In [752]:
gdf_F_idf_4patho = gdf_patientes_idf[gdf_patientes_idf["patho"].isin(pathologie)]
print(f"{(len(gdf_F_idf_4patho)/len(gdf_patientes_idf)*100).__round__(2)}% des patientes en IDF sont atteints de cancer de Sein, Hemato, Gynéco ou Thorax")
len(gdf_F_idf_4patho)

89.94% des patientes en IDF sont atteints de cancer de Sein, Hemato, Gynéco ou Thorax


32213

In [753]:
# NOMBRE DE PATIENTES PAR PATHOLOGIE EN IDF
tab=[]
for patho in pathologie:
    df_patho = gdf_F_idf_4patho[gdf_F_idf_4patho['patho']== patho]
    tab.append({"patho": patho , "nb_patients_co": len(df_patho)})
tab = pd.DataFrame(tab)
print(tab)

print(sum(tab['nb_patients_co']))




    patho  nb_patients_co
0  Hemato             952
1    Sein           28625
2  Gynéco            1939
3  Thorax             697
32213


# STATISTIQUES EN FONCTION DE LA PROXIMITE AUX VOIES FERREES EN IDF

In [754]:
def filter_and_group_by_patho(gdf, patho, column_name):
    gdf_patho = gdf[gdf['patho'] == patho]
    gdf_patho_grouped = gdf_patho.groupby('CODE_DEPT').size().reset_index(name=column_name)
    return gdf_patho_grouped

In [755]:
gdf_idf_hemato = filter_and_group_by_patho(gdf_patientes_idf, 'Hemato', 'nb_patientes_hemato')
gdf_idf_sein = filter_and_group_by_patho(gdf_patientes_idf, 'Sein', 'nb_patientes_sein')
gdf_idf_gyneco = filter_and_group_by_patho(gdf_patientes_idf, 'Gynéco', 'nb_patientes_gyneco')
gdf_idf_thorax = filter_and_group_by_patho(gdf_patientes_idf, 'Thorax', 'nb_patientes_thorax')

gdf_idf_thorax

,CODE_DEPT,nb_patientes_thorax
0,75,227
1,77,56
2,78,81
3,91,24
4,92,171
5,93,47
6,94,58
7,95,33


### DANS UN PERIMETRE DE 150M 

#### 1- Repartition de l'ensemble des patients dans cette zone

In [756]:
# df_patients_150m = pd.read_csv("Resultats/Excels/patients_FR_IDF_geocoded_adultes_clinique_150m.csv", sep=";")
# df_patients_150m["CODE_DEPT_"] = df_patients_150m["CODE_DEPT_"].astype(str)
# df_patientes_150m = df_patients_150m[df_patients_150m['CODE_DEPT_'].isin(dep_idf)]
# df_patientes_150m = df_patientes_150m[df_patientes_150m["patient_se"]=="F"]
# df_patientes_150m.columns


In [757]:
gdf_buffer_150m = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/CARTO_QGIS/voies_ferrees_idf/zone_150m.shp')
gdf_buffer_150m = gdf_buffer_150m.to_crs(gdf_patients.crs)
gdf_buffer_150m.columns

Index(['ID', 'ETAT', 'NATURE', 'ELECTRIFIE', 'NB_VOIES', 'LARGEUR', 'POS_SOL',
       'ID_VFN', 'TOPONYME', 'layer', 'path', 'geometry'],
      dtype='object')

In [758]:
df_patientes_150m_4patho = gpd.sjoin(gdf_F_idf_4patho.to_crs(gdf_buffer_150m.crs), gdf_buffer_150m, how='inner', predicate='intersects')

In [759]:
pourcentage_150m = ((len(df_patientes_150m_4patho.drop_duplicates(subset = ['pseudo_provisoire'])))/len(gdf_patientes_idf.drop_duplicates(subset=['pseudo_provisoire']))*100)
print(f"{pourcentage_150m:.2f} % de patientes sont situées à moins de 150m des voies ferrées en IDF")

12.42 % de patientes sont situées à moins de 150m des voies ferrées en IDF


Repartition par departement

In [760]:
df_patientes_count_dept = gdf_F_idf_4patho.groupby("CODE_DEPT").size().reset_index(name = 'total')
gdf_dept = gdf_dept[["CODE_DEPT", "NOM_DEPT"]]
patientes_counts_by_dept = gdf_dept.merge(df_patientes_count_dept, on = 'CODE_DEPT')
patientes_counts_by_dept.sort_values(by='CODE_DEPT')

,CODE_DEPT,NOM_DEPT,total
3,75,PARIS,7508
5,77,SEINE-ET-MARNE,2300
1,78,YVELINES,6383
6,91,ESSONNE,2168
2,92,HAUTS-DE-SEINE,7153
0,93,SEINE-SAINT-DENIS,2410
4,94,VAL-DE-MARNE,2209
7,95,VAL-D'OISE,2082


In [761]:
sum(df_patientes_count_dept["total"])

32213

In [762]:
patientes_counts_exp_by_dept = df_patientes_150m_4patho[["CODE_DEPT"]].groupby('CODE_DEPT').size().reset_index(name = 'exposes')
patientes_counts_exp_by_dept

,CODE_DEPT,exposes
0,75,1455
1,77,167
2,78,645
3,91,202
4,92,1187
5,93,251
6,94,294
7,95,248


In [763]:
stats_150m = patientes_counts_by_dept.merge(patientes_counts_exp_by_dept, left_on='CODE_DEPT', right_on='CODE_DEPT')
#stats_150m.drop(columns=['CODE_DEPT'], inplace=True)
stats_150m.sort_values(by='CODE_DEPT')


,CODE_DEPT,NOM_DEPT,total,exposes
3,75,PARIS,7508,1455
5,77,SEINE-ET-MARNE,2300,167
1,78,YVELINES,6383,645
6,91,ESSONNE,2168,202
2,92,HAUTS-DE-SEINE,7153,1187
0,93,SEINE-SAINT-DENIS,2410,251
4,94,VAL-DE-MARNE,2209,294
7,95,VAL-D'OISE,2082,248


In [764]:
stats_150m["Pourcentage_150"] = (stats_150m["exposes"]/stats_150m["total"]*100).round(2)
stats_150m.sort_values(by='CODE_DEPT')

,CODE_DEPT,NOM_DEPT,total,exposes,Pourcentage_150
3,75,PARIS,7508,1455,19.38
5,77,SEINE-ET-MARNE,2300,167,7.26
1,78,YVELINES,6383,645,10.10
6,91,ESSONNE,2168,202,9.32
2,92,HAUTS-DE-SEINE,7153,1187,16.59
0,93,SEINE-SAINT-DENIS,2410,251,10.41
4,94,VAL-DE-MARNE,2209,294,13.31
7,95,VAL-D'OISE,2082,248,11.91


In [765]:
fig = px.bar(stats_150m, x='NOM_DEPT', y='Pourcentage_150', title='Patientèle par departement située à une distance maximale de 150m des voies ferrées',
             labels={'NOM_DEPT': 'Département', 'Pourcentage_150': 'patients(%)'})

fig.show()

#### par pathologie

In [766]:
#nombre de patients par departement et par pathologie
gdf_idf_hemato_dept = gdf_idf_hemato.merge(gdf_dept[['NOM_DEPT','CODE_DEPT']], on='CODE_DEPT')
gdf_idf_sein_dept = gdf_idf_sein.merge(gdf_dept[['NOM_DEPT','CODE_DEPT']], on='CODE_DEPT')
gdf_idf_gyneco_dept = gdf_idf_gyneco.merge(gdf_dept[['NOM_DEPT','CODE_DEPT']], on='CODE_DEPT')
gdf_idf_thorax_dept = gdf_idf_thorax.merge(gdf_dept[['NOM_DEPT','CODE_DEPT']], on='CODE_DEPT')

gdf_idf_thorax_dept

,CODE_DEPT,nb_patientes_thorax,NOM_DEPT
0,75,227,PARIS
1,77,56,SEINE-ET-MARNE
2,78,81,YVELINES
3,91,24,ESSONNE
4,92,171,HAUTS-DE-SEINE
5,93,47,SEINE-SAINT-DENIS
6,94,58,VAL-DE-MARNE
7,95,33,VAL-D'OISE


In [767]:
#patients par pathologie
gdf_idf_hemato_points = gdf_patientes_idf[gdf_patientes_idf['patho'] == 'Hemato'][['CODE_DEPT', 'geometry']]
gdf_idf_sein_points = gdf_patientes_idf[gdf_patientes_idf['patho'] == 'Sein'][['CODE_DEPT', 'geometry']]
gdf_idf_gyneco_points = gdf_patientes_idf[gdf_patientes_idf['patho'] == 'Gynéco'][['CODE_DEPT', 'geometry']]
gdf_idf_thorax_points = gdf_patientes_idf[gdf_patientes_idf['patho'] == 'Thorax'][['CODE_DEPT', 'geometry']]




In [768]:
#nombre de patients par pathologie situes a 150m
gdf_idf_hemato_150m = gpd.sjoin(gdf_idf_hemato_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_hemato_150m_dept = gdf_idf_hemato_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_hemato_150m = gdf_idf_hemato_150m_dept.merge(gdf_idf_hemato_dept, on='CODE_DEPT')
gdf_idf_hemato_150m['pourcentage']= (gdf_idf_hemato_150m['nbre_patients_<=150m']/gdf_idf_hemato_150m['nb_patientes_hemato']*100).round(1)


gdf_idf_sein_150m = gpd.sjoin(gdf_idf_sein_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_sein_150m_dept = gdf_idf_sein_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_sein_150m = gdf_idf_sein_150m_dept.merge(gdf_idf_sein_dept, on='CODE_DEPT')
gdf_idf_sein_150m['pourcentage']= (gdf_idf_sein_150m['nbre_patients_<=150m']/gdf_idf_sein_150m['nb_patientes_sein']*100).round(1)


gdf_idf_thorax_150m = gpd.sjoin(gdf_idf_thorax_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_thorax_150m_dept = gdf_idf_thorax_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_thorax_150m = gdf_idf_thorax_150m_dept.merge(gdf_idf_thorax_dept, on='CODE_DEPT')
gdf_idf_thorax_150m['pourcentage']= (gdf_idf_thorax_150m['nbre_patients_<=150m']/gdf_idf_thorax_150m['nb_patientes_thorax']*100).round(1)


gdf_idf_gyneco_150m = gpd.sjoin(gdf_idf_gyneco_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_gyneco_150m_dept = gdf_idf_gyneco_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_gyneco_150m = gdf_idf_gyneco_150m_dept.merge(gdf_idf_gyneco_dept, on='CODE_DEPT')
gdf_idf_gyneco_150m['pourcentage'] = (gdf_idf_gyneco_150m['nbre_patients_<=150m']/gdf_idf_gyneco_150m['nb_patientes_gyneco']*100).round(1)

gdf_idf_gyneco_150m


,CODE_DEPT,nbre_patients_<=150m,nb_patientes_gyneco,NOM_DEPT,pourcentage
0,75,60,373,PARIS,16.1
1,77,9,121,SEINE-ET-MARNE,7.4
2,78,30,352,YVELINES,8.5
3,91,14,120,ESSONNE,11.7
4,92,87,518,HAUTS-DE-SEINE,16.8
5,93,21,214,SEINE-SAINT-DENIS,9.8
6,94,12,107,VAL-DE-MARNE,11.2
7,95,9,134,VAL-D'OISE,6.7


In [769]:

# Création d'une figure avec des sous-trames
fig =  go.Figure()

fig.add_trace(go.Bar(x=gdf_idf_hemato_150m['NOM_DEPT'], y=gdf_idf_hemato_150m['pourcentage'], name='cancer hemato',
                     text=gdf_idf_hemato_150m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_sein_150m['NOM_DEPT'], y=gdf_idf_sein_150m['pourcentage'], name='cancer sein',
                     text=gdf_idf_sein_150m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_thorax_150m['NOM_DEPT'], y=gdf_idf_thorax_150m['pourcentage'], name='cancer thorax',
                     text=gdf_idf_thorax_150m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_gyneco_150m['NOM_DEPT'], y=gdf_idf_gyneco_150m['pourcentage'], name='cancer Gynéco',
                     text=gdf_idf_gyneco_150m['pourcentage'].round(1),
                     textposition='outside'))
#fig.add_trace(go.Bar(x=gdf_idf_thorax_150m['NOM_DEPT'], y=gdf_idf_thorax_150m['pourcentage'], name='cancer thorax',
                     #text=gdf_idf_thorax_150m['pourcentage'].round(1),
                     #textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Département'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle repartie par département et située à une distance maximale de 150m des voies ferrées en fonction de leur groupe pathologique')
config = {
    'toImageButtonOptions': {
        'format': 'png',  # one of png, svg, jpeg, webp
        'filename': 'bar_stats_150m',
        'height': 1080,
        'width': 1920,
        'scale': 5  # Multiply title/legend/axis/canvas sizes by this factor
    }
}

fig.show(config=config)

### DANS UN PERIMETRE DE 500M 

In [770]:
# df_patients_500m = pd.read_csv("Resultats/Excels/patients_FR_IDF_geocoded_adultes_clinique_500m.csv", sep=";")
# df_patients_500m["CODE_DEPT_"] = df_patients_500m["CODE_DEPT_"].astype(str)
# df_patientes_500m = df_patients_500m[df_patients_500m['CODE_DEPT_'].isin(dep_idf)]
# df_patientes_500m = df_patientes_500m[df_patientes_500m["patient_se"]=="F"]
# df_patientes_500m.columns

In [771]:
gdf_buffer_500m = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/CARTO_QGIS/voies_ferrees_idf/zone_500m.shp')
gdf_buffer_500m = gdf_buffer_500m.to_crs(gdf_patients.crs)
gdf_buffer_500m.columns

Index(['ID', 'ETAT', 'NATURE', 'ELECTRIFIE', 'NB_VOIES', 'LARGEUR', 'POS_SOL',
       'ID_VFN', 'TOPONYME', 'layer', 'path', 'geometry'],
      dtype='object')

In [772]:
df_patientes_500m_4patho = gpd.sjoin(gdf_F_idf_4patho.to_crs(gdf_buffer_500m.crs), gdf_buffer_500m, how='inner', predicate='intersects')
len(df_patientes_500m_4patho)

14569

In [773]:
pourcentage_500m = ((len(df_patientes_500m_4patho.drop_duplicates(subset = ['pseudo_provisoire'])))/len(gdf_patientes_idf.drop_duplicates(subset=['pseudo_provisoire']))*100)
print(f"{pourcentage_500m:.2f} % de patientes sont situées à moins de 500m des voies ferrées en IDF")

40.68 % de patientes sont situées à moins de 500m des voies ferrées en IDF


In [774]:
patientes_counts_exp_by_dept = df_patientes_500m_4patho[["CODE_DEPT"]].groupby('CODE_DEPT').size().reset_index(name = 'exposes')
patientes_counts_exp_by_dept

,CODE_DEPT,exposes
0,75,4577
1,77,708
2,78,2195
3,91,726
4,92,3501
5,93,972
6,94,1006
7,95,884


In [775]:
stats_500m = patientes_counts_by_dept.merge(patientes_counts_exp_by_dept, left_on='CODE_DEPT', right_on='CODE_DEPT')
stats_500m.sort_values(by='CODE_DEPT')
stats_500m["Pourcentage_500"] = (stats_500m["exposes"]/stats_500m["total"]*100).round(2)
stats_500m.sort_values(by='CODE_DEPT')




,CODE_DEPT,NOM_DEPT,total,exposes,Pourcentage_500
3,75,PARIS,7508,4577,60.96
5,77,SEINE-ET-MARNE,2300,708,30.78
1,78,YVELINES,6383,2195,34.39
6,91,ESSONNE,2168,726,33.49
2,92,HAUTS-DE-SEINE,7153,3501,48.94
0,93,SEINE-SAINT-DENIS,2410,972,40.33
4,94,VAL-DE-MARNE,2209,1006,45.54
7,95,VAL-D'OISE,2082,884,42.46


In [776]:
fig = px.bar(stats_500m, x='NOM_DEPT', y='Pourcentage_500', title='Patientèle par departement située à une distance maximale de 500m des voies ferrées',
             labels={'NOM_DEPT': 'Département', 'Pourcentage_500': 'patients(%)'})
fig.show()

Par pathologie

In [777]:
gdf_idf_hemato_500m = gpd.sjoin(gdf_idf_hemato_points, gdf_buffer_500m, how='inner', predicate='intersects')
gdf_idf_hemato_500m_dept = gdf_idf_hemato_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_hemato_500m = gdf_idf_hemato_500m_dept.merge(gdf_idf_hemato_dept, on='CODE_DEPT')
gdf_idf_hemato_500m['pourcentage']= (gdf_idf_hemato_500m['nbre_patients_<=500m']/gdf_idf_hemato_500m['nb_patientes_hemato']*100).round(0)


gdf_idf_sein_500m = gpd.sjoin(gdf_idf_sein_points, gdf_buffer_500m, how='inner', predicate='intersects')
gdf_idf_sein_500m_dept = gdf_idf_sein_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_sein_500m = gdf_idf_sein_500m_dept.merge(gdf_idf_sein_dept, on='CODE_DEPT')
gdf_idf_sein_500m['pourcentage']= (gdf_idf_sein_500m['nbre_patients_<=500m']/gdf_idf_sein_500m['nb_patientes_sein']*100).round(0)


gdf_idf_thorax_500m = gpd.sjoin(gdf_idf_thorax_points, gdf_buffer_500m, how='inner', predicate='intersects')
gdf_idf_thorax_500m_dept = gdf_idf_thorax_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_thorax_500m = gdf_idf_thorax_500m_dept.merge(gdf_idf_thorax_dept, on='CODE_DEPT')
gdf_idf_thorax_500m['pourcentage']= (gdf_idf_thorax_500m['nbre_patients_<=500m']/gdf_idf_thorax_500m['nb_patientes_thorax']*100).round(0)


gdf_idf_gyneco_500m = gpd.sjoin(gdf_idf_gyneco_points, gdf_buffer_500m, how='inner', predicate='intersects')
gdf_idf_gyneco_500m_dept = gdf_idf_gyneco_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_gyneco_500m = gdf_idf_gyneco_500m_dept.merge(gdf_idf_gyneco_dept, on='CODE_DEPT')
gdf_idf_gyneco_500m['pourcentage'] = (gdf_idf_gyneco_500m['nbre_patients_<=500m']/gdf_idf_gyneco_150m['nb_patientes_gyneco']*100).round(0)

gdf_idf_hemato_500m


,CODE_DEPT,nbre_patients_<=500m,nb_patientes_hemato,NOM_DEPT,pourcentage
0,75,112,193,PARIS,58.0
1,77,14,39,SEINE-ET-MARNE,36.0
2,78,57,154,YVELINES,37.0
3,91,18,50,ESSONNE,36.0
4,92,193,365,HAUTS-DE-SEINE,53.0
5,93,13,47,SEINE-SAINT-DENIS,28.0
6,94,17,50,VAL-DE-MARNE,34.0
7,95,13,54,VAL-D'OISE,24.0


In [778]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_hemato_500m['NOM_DEPT'], y=gdf_idf_hemato_500m['pourcentage'], name='cancer hemato',
                     text=gdf_idf_hemato_500m['pourcentage'],
                     textposition='outside',textfont=dict(size=14)))
fig.add_trace(go.Bar(x=gdf_idf_sein_500m['NOM_DEPT'], y=gdf_idf_sein_500m['pourcentage'], name='cancer sein',
                     text=gdf_idf_sein_500m['pourcentage'],
                     textposition='outside',textfont=dict(size=14)))
fig.add_trace(go.Bar(x=gdf_idf_thorax_500m['NOM_DEPT'], y=gdf_idf_thorax_500m['pourcentage'], name='cancer thorax',
                     text=gdf_idf_thorax_500m['pourcentage'],
                     textposition='outside',textfont=dict(size=14)))
fig.add_trace(go.Bar(x=gdf_idf_gyneco_500m['NOM_DEPT'], y=gdf_idf_gyneco_500m['pourcentage'], name='cancer gyneco',
                     text=gdf_idf_gyneco_500m['pourcentage'],
                     textposition='outside',textfont=dict(size=14)))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Département'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle située à une distance maximale de 500m des voies ferrées par département et en fonction de leur pathologie')

config = {
    'toImageButtonOptions': {
        'format': 'png',  # one of png, svg, jpeg, webp
        'filename': 'bar_stats_500m',
        'height': 500,
        'width': 1500,
        'scale': 5  # Multiply title/legend/axis/canvas sizes by this factor
    }
}

fig.show(config=config)

### DANS UN PERIMETRE DE 1KM 

In [779]:
gdf_buffer_1km = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/CARTO_QGIS/voies_ferrees_idf/zone_1km.shp')
gdf_buffer_1km = gdf_buffer_1km.to_crs(gdf_patients.crs)
gdf_buffer_1km.columns

Index(['ID', 'ETAT', 'NATURE', 'ELECTRIFIE', 'NB_VOIES', 'LARGEUR', 'POS_SOL',
       'ID_VFN', 'TOPONYME', 'layer', 'path', 'geometry'],
      dtype='object')

In [780]:
df_patientes_1km_4patho = gpd.sjoin(gdf_F_idf_4patho.to_crs(gdf_buffer_1km.crs), gdf_buffer_1km, how='inner', predicate='intersects')
len(df_patientes_1km_4patho)

23516

In [781]:
pourcentage_1km = ((len(df_patientes_1km_4patho.drop_duplicates(subset = ['pseudo_provisoire'])))/len(gdf_patientes_idf.drop_duplicates(subset=['pseudo_provisoire']))*100)
print(f"{pourcentage_1km:.2f} % de patientes sont situées à moins d\'un km des voies ferrées en IDF")

65.65 % de patientes sont situées à moins d'un km des voies ferrées en IDF


In [782]:
patientes_counts_exp_by_dept = df_patientes_1km_4patho[["CODE_DEPT"]].groupby('CODE_DEPT').size().reset_index(name = 'exposes')
patientes_counts_exp_by_dept
stats_1km = patientes_counts_by_dept.merge(patientes_counts_exp_by_dept, left_on='CODE_DEPT', right_on='CODE_DEPT')
stats_1km.sort_values(by='CODE_DEPT')
stats_1km["Pourcentage_1km"] = (stats_1km["exposes"]/stats_1km["total"]*100).round(2)
stats_1km.sort_values(by='CODE_DEPT')


,CODE_DEPT,NOM_DEPT,total,exposes,Pourcentage_1km
3,75,PARIS,7508,6951,92.58
5,77,SEINE-ET-MARNE,2300,1221,53.09
1,78,YVELINES,6383,3743,58.64
6,91,ESSONNE,2168,1288,59.41
2,92,HAUTS-DE-SEINE,7153,5564,77.79
0,93,SEINE-SAINT-DENIS,2410,1652,68.55
4,94,VAL-DE-MARNE,2209,1665,75.37
7,95,VAL-D'OISE,2082,1432,68.78


In [783]:
fig = px.bar(stats_1km, x='NOM_DEPT', y='Pourcentage_1km', title='Patientèle par departement située à une distance maximale d\'un km des voies ferrées',
             labels={'NOM_DEPT': 'Département', 'Pourcentage_1km': 'patients(%)'})
fig.show()

par pathologie

In [784]:
gdf_idf_hemato_1km = gpd.sjoin(gdf_idf_hemato_points, gdf_buffer_1km, how='inner', predicate='intersects')
gdf_idf_hemato_1km_dept = gdf_idf_hemato_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_hemato_1km = gdf_idf_hemato_1km_dept.merge(gdf_idf_hemato_dept, on='CODE_DEPT')
gdf_idf_hemato_1km['pourcentage']= (gdf_idf_hemato_1km['nbre_patients_<=1km']/gdf_idf_hemato_1km['nb_patientes_hemato']*100).round(0)


gdf_idf_sein_1km = gpd.sjoin(gdf_idf_sein_points, gdf_buffer_1km, how='inner', predicate='intersects')
gdf_idf_sein_1km_dept = gdf_idf_sein_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_sein_1km = gdf_idf_sein_1km_dept.merge(gdf_idf_sein_dept, on='CODE_DEPT')
gdf_idf_sein_1km['pourcentage']= (gdf_idf_sein_1km['nbre_patients_<=1km']/gdf_idf_sein_1km['nb_patientes_sein']*100).round(0)


gdf_idf_thorax_1km = gpd.sjoin(gdf_idf_thorax_points, gdf_buffer_1km, how='inner', predicate='intersects')
gdf_idf_thorax_1km_dept = gdf_idf_thorax_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_thorax_1km = gdf_idf_thorax_1km_dept.merge(gdf_idf_thorax_dept, on='CODE_DEPT')
gdf_idf_thorax_1km['pourcentage']= (gdf_idf_thorax_1km['nbre_patients_<=1km']/gdf_idf_thorax_1km['nb_patientes_thorax']*100).round(0)


gdf_idf_gyneco_1km = gpd.sjoin(gdf_idf_gyneco_points, gdf_buffer_1km, how='inner', predicate='intersects')
gdf_idf_gyneco_1km_dept = gdf_idf_gyneco_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_gyneco_1km = gdf_idf_gyneco_1km_dept.merge(gdf_idf_gyneco_dept, on='CODE_DEPT')
gdf_idf_gyneco_1km['pourcentage'] = (gdf_idf_gyneco_1km['nbre_patients_<=1km']/gdf_idf_gyneco_1km['nb_patientes_gyneco']*100).round(0)

gdf_idf_gyneco_1km

,CODE_DEPT,nbre_patients_<=1km,nb_patientes_gyneco,NOM_DEPT,pourcentage
0,75,334,373,PARIS,90.0
1,77,67,121,SEINE-ET-MARNE,55.0
2,78,209,352,YVELINES,59.0
3,91,66,120,ESSONNE,55.0
4,92,410,518,HAUTS-DE-SEINE,79.0
5,93,161,214,SEINE-SAINT-DENIS,75.0
6,94,88,107,VAL-DE-MARNE,82.0
7,95,87,134,VAL-D'OISE,65.0


In [ ]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_hemato_1km['NOM_DEPT'], y=gdf_idf_hemato_1km['pourcentage'], name='cancer hemato',
                     text=gdf_idf_hemato_1km['pourcentage'],
                     textposition='outside',textfont=dict(size=16)))
fig.add_trace(go.Bar(x=gdf_idf_sein_1km['NOM_DEPT'], y=gdf_idf_sein_1km['pourcentage'], name='cancer sein',
                     text=gdf_idf_sein_1km['pourcentage'],
                     textposition='outside',textfont=dict(size=16)))
fig.add_trace(go.Bar(x=gdf_idf_thorax_1km['NOM_DEPT'], y=gdf_idf_thorax_1km['pourcentage'], name='cancer Thorax',
                     text=gdf_idf_thorax_1km['pourcentage'],
                     textposition='outside',textfont=dict(size=16)))
fig.add_trace(go.Bar(x=gdf_idf_gyneco_1km['NOM_DEPT'], y=gdf_idf_gyneco_1km['pourcentage'], name='cancer Gynéco',
                     text=gdf_idf_gyneco_1km['pourcentage'],
                     textposition='outside',textfont=dict(size=16)))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Département'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle située à une distance maximale d\'1km des voies ferrées par département et en fonction de leur pathologie')

config = {
    'toImageButtonOptions': {
        'format': 'png', 
        'filename': 'bar_stats_1000m',
        'height': 500,
        'width': 1500,
        'scale': 5  # Multiply title/legend/axis/canvas sizes by this factor
    }
}

fig.show(config=config)

#### Perimetre de 2.5km

In [786]:
gdf_buffer_2_5km = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/CARTO_QGIS/voies_ferrees_idf/zone_2.5_km.shp')
gdf_buffer_2_5km = gdf_buffer_2_5km.to_crs(gdf_patients.crs)


In [787]:
df_patientes_2_5km_4patho = gpd.sjoin(gdf_F_idf_4patho.to_crs(gdf_buffer_2_5km.crs), gdf_buffer_2_5km, how='inner', predicate='intersects')
pourcentage_2_5km = ((len(df_patientes_2_5km_4patho.drop_duplicates(subset = ['pseudo_provisoire'])))/len(gdf_patientes_idf.drop_duplicates(subset=['pseudo_provisoire']))*100)
print(len(df_patientes_2_5km_4patho))
print(f"{pourcentage_2_5km:.2f} % de patientes sont situées à moins de 2.5 km des voies ferrées en IDF")


30695
85.70 % de patientes sont situées à moins de 2.5 km des voies ferrées en IDF


In [788]:
patientes_counts_exp_by_dept = df_patientes_2_5km_4patho[["CODE_DEPT"]].groupby('CODE_DEPT').size().reset_index(name = 'exposes')
patientes_counts_exp_by_dept
stats_2_5km = patientes_counts_by_dept.merge(patientes_counts_exp_by_dept, left_on='CODE_DEPT', right_on='CODE_DEPT')
stats_2_5km.sort_values(by='CODE_DEPT')
stats_2_5km["Pourcentage_2_5km"] = (stats_2_5km["exposes"]/stats_2_5km["total"]*100).round(2)
stats_2_5km = stats_2_5km.sort_values(by='CODE_DEPT')
stats_2_5km


,CODE_DEPT,NOM_DEPT,total,exposes,Pourcentage_2_5km
3,75,PARIS,7508,7508,100.00
5,77,SEINE-ET-MARNE,2300,1914,83.22
1,78,YVELINES,6383,5704,89.36
6,91,ESSONNE,2168,1963,90.54
2,92,HAUTS-DE-SEINE,7153,7141,99.83
0,93,SEINE-SAINT-DENIS,2410,2357,97.80
4,94,VAL-DE-MARNE,2209,2147,97.19
7,95,VAL-D'OISE,2082,1961,94.19


In [789]:
fig = px.bar(stats_2_5km, x='NOM_DEPT', y='Pourcentage_2_5km', title='Patientèle par departement située à une distance maximale de 2.5 km des voies ferrées',
             labels={'NOM_DEPT': 'Département', 'Pourcentage_2_5km': 'patients(%)'})
fig.show()

In [790]:
gdf_idf_hemato_2_5km = gpd.sjoin(gdf_idf_hemato_points, gdf_buffer_2_5km, how='inner', predicate='intersects')
gdf_idf_hemato_2_5km_dept = gdf_idf_hemato_2_5km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2.5km')
gdf_idf_hemato_2_5km = gdf_idf_hemato_2_5km_dept.merge(gdf_idf_hemato_dept, on='CODE_DEPT')
gdf_idf_hemato_2_5km['pourcentage']= (gdf_idf_hemato_2_5km['nbre_patients_<=2.5km']/gdf_idf_hemato_2_5km['nb_patientes_hemato']*100).round(1)


gdf_idf_sein_2_5km = gpd.sjoin(gdf_idf_sein_points, gdf_buffer_2_5km, how='inner', predicate='intersects')
gdf_idf_sein_2_5km_dept = gdf_idf_sein_2_5km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2.5km')
gdf_idf_sein_2_5km = gdf_idf_sein_2_5km_dept.merge(gdf_idf_sein_dept, on='CODE_DEPT')
gdf_idf_sein_2_5km['pourcentage']= (gdf_idf_sein_2_5km['nbre_patients_<=2.5km']/gdf_idf_sein_2_5km['nb_patientes_sein']*100).round(1)


gdf_idf_thorax_2_5km = gpd.sjoin(gdf_idf_thorax_points, gdf_buffer_2_5km, how='inner', predicate='intersects')
gdf_idf_thorax_2_5km_dept = gdf_idf_thorax_2_5km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2.5km')
gdf_idf_thorax_2_5km = gdf_idf_thorax_2_5km_dept.merge(gdf_idf_thorax_dept, on='CODE_DEPT')
gdf_idf_thorax_2_5km['pourcentage']= (gdf_idf_thorax_2_5km['nbre_patients_<=2.5km']/gdf_idf_thorax_2_5km['nb_patientes_thorax']*100).round(1)


gdf_idf_gyneco_2_5km = gpd.sjoin(gdf_idf_gyneco_points, gdf_buffer_2_5km, how='inner', predicate='intersects')
gdf_idf_gyneco_2_5km_dept = gdf_idf_gyneco_2_5km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2.5km')
gdf_idf_gyneco_2_5km = gdf_idf_gyneco_2_5km_dept.merge(gdf_idf_gyneco_dept, on='CODE_DEPT')
gdf_idf_gyneco_2_5km['pourcentage'] = (gdf_idf_gyneco_2_5km['nbre_patients_<=2.5km']/gdf_idf_gyneco_2_5km['nb_patientes_gyneco']*100).round(1)

gdf_idf_gyneco_2_5km

,CODE_DEPT,nbre_patients_<=2.5km,nb_patientes_gyneco,NOM_DEPT,pourcentage
0,75,373,373,PARIS,100.0
1,77,94,121,SEINE-ET-MARNE,77.7
2,78,317,352,YVELINES,90.1
3,91,111,120,ESSONNE,92.5
4,92,518,518,HAUTS-DE-SEINE,100.0
5,93,210,214,SEINE-SAINT-DENIS,98.1
6,94,106,107,VAL-DE-MARNE,99.1
7,95,126,134,VAL-D'OISE,94.0


In [791]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_hemato_2_5km['NOM_DEPT'], y=gdf_idf_hemato_2_5km['pourcentage'], name='cancer hemato',
                     text=gdf_idf_hemato_2_5km['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_sein_2_5km['NOM_DEPT'], y=gdf_idf_sein_2_5km['pourcentage'], name='cancer sein',
                     text=gdf_idf_sein_2_5km['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_thorax_2_5km['NOM_DEPT'], y=gdf_idf_thorax_2_5km['pourcentage'], name='cancer Thorax',
                     text=gdf_idf_thorax_2_5km['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_gyneco_2_5km['NOM_DEPT'], y=gdf_idf_gyneco_2_5km['pourcentage'], name='cancer Gynéco',
                     text=gdf_idf_gyneco_2_5km['pourcentage'],
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Département'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle située à une distance maximale de 2.5 km des voies ferrées par département et en fonction de leur pathologie')

fig.show()

# STATISTIQUES EN FONCTION DE LA PROXIMITE AUX VOIES ROUTIERES PRINCIPALES EN IDF

In [792]:
gdf_buffer_150m = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/CARTO_QGIS/highways/highways_150m.shp')
gdf_buffer_150m = gdf_buffer_150m.to_crs(gdf_patients.crs)

gdf_buffer_500m = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/CARTO_QGIS/highways/highways_500m.shp')
gdf_buffer_500m = gdf_buffer_500m.to_crs(gdf_patients.crs)

gdf_buffer_1km = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/CARTO_QGIS/highways/highways_1km.shp')
gdf_buffer_1km = gdf_buffer_1km.to_crs(gdf_patients.crs)

In [793]:
gdf_buffer_150m = gdf_buffer_150m.to_crs(gdf_patients.crs)

In [794]:
#Toutes pathologies confondus 

gdf_patients_150m = pd.read_csv("Resultats/Excels/patients_highways/patients_FR_IDF_geocoded_adultes_clinique_150m.csv",sep=";")
gdf_patientes_150m = gdf_patients_150m[gdf_patients_150m["patient_sexe"]=="F"]

gdf_patients_500m = pd.read_csv("Resultats/Excels/patients_highways/patients_FR_IDF_geocoded_adultes_clinique_500m.csv",sep=";")
gdf_patientes_500m = gdf_patients_500m[gdf_patients_500m["patient_sexe"]=="F"] 

gdf_patients_1km = pd.read_csv("Resultats/Excels/patients_highways/patients_FR_IDF_geocoded_adultes_clinique_1km.csv",sep=";")
gdf_patientes_1km = gdf_patients_1km[gdf_patients_1km["patient_sexe"]=="F"] 

In [795]:
print(len(gdf_patientes_150m))
print(len(gdf_patientes_500m))
print(len(gdf_patientes_1km))

841
5389
12706


In [796]:
#En  considerant uniquement les 4 pathologies

gdf_F_4patho_150m = gpd.sjoin(gdf_F_idf_4patho.to_crs(gdf_buffer_150m.crs), gdf_buffer_150m, how='inner', predicate='intersects')
gdf_F_4patho_500m = gpd.sjoin(gdf_F_idf_4patho.to_crs(gdf_buffer_500m.crs), gdf_buffer_500m, how='inner', predicate='intersects')
gdf_F_4patho_1km = gpd.sjoin(gdf_F_idf_4patho.to_crs(gdf_buffer_1km.crs), gdf_buffer_1km, how='inner', predicate='intersects')

print(len(gdf_F_4patho_150m))
print(len(gdf_F_4patho_500m))
print(len(gdf_F_4patho_1km))


746
4838
11406


### DANS UN PERIMETRE DE 150M 

In [797]:
pourcentage_150m = ((len(gdf_F_4patho_150m.drop_duplicates(subset = ['pseudo_provisoire'])))/len(gdf_patientes_idf.drop_duplicates(subset=['pseudo_provisoire']))*100)
print(f"{pourcentage_150m:.2f} % de patientes sont situées à moins de 150m des voies routières principales en IDF")

2.08 % de patientes sont situées à moins de 150m des voies routières principales en IDF


In [798]:
patientes_counts_exp_by_dept = gdf_F_4patho_150m[["CODE_DEPT"]].groupby('CODE_DEPT').size().reset_index(name = 'exposes')
patientes_counts_exp_by_dept
stats_150m = patientes_counts_by_dept.merge(patientes_counts_exp_by_dept, left_on='CODE_DEPT', right_on='CODE_DEPT')
stats_150m.sort_values(by='CODE_DEPT')
stats_150m["Pourcentage_150"] = (stats_150m["exposes"]/stats_150m["total"]*100).round(2)
stats_150m.sort_values(by='CODE_DEPT')



,CODE_DEPT,NOM_DEPT,total,exposes,Pourcentage_150
3,75,PARIS,7508,105,1.40
5,77,SEINE-ET-MARNE,2300,35,1.52
1,78,YVELINES,6383,93,1.46
6,91,ESSONNE,2168,28,1.29
2,92,HAUTS-DE-SEINE,7153,269,3.76
0,93,SEINE-SAINT-DENIS,2410,79,3.28
4,94,VAL-DE-MARNE,2209,113,5.12
7,95,VAL-D'OISE,2082,24,1.15


In [799]:
fig = px.bar(stats_150m, x='NOM_DEPT', y='Pourcentage_150', title='Patientèle par departement située à une distance maximale de 150m des voies routières principales',
             labels={'NOM_DEPT': 'Département', 'Pourcentage_150': 'patients(%)'})
fig.show()

In [800]:
#nombre de patients par pathologie situes a 150m
gdf_idf_hemato_150m = gpd.sjoin(gdf_idf_hemato_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_hemato_150m_dept = gdf_idf_hemato_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_hemato_150m = gdf_idf_hemato_150m_dept.merge(gdf_idf_hemato_dept, on='CODE_DEPT')
gdf_idf_hemato_150m['pourcentage']= (gdf_idf_hemato_150m['nbre_patients_<=150m']/gdf_idf_hemato_150m['nb_patientes_hemato']*100).round(1)


gdf_idf_sein_150m = gpd.sjoin(gdf_idf_sein_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_sein_150m_dept = gdf_idf_sein_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_sein_150m = gdf_idf_sein_150m_dept.merge(gdf_idf_sein_dept, on='CODE_DEPT')
gdf_idf_sein_150m['pourcentage']= (gdf_idf_sein_150m['nbre_patients_<=150m']/gdf_idf_sein_150m['nb_patientes_sein']*100).round(1)


gdf_idf_thorax_150m = gpd.sjoin(gdf_idf_thorax_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_thorax_150m_dept = gdf_idf_thorax_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_thorax_150m = gdf_idf_thorax_150m_dept.merge(gdf_idf_thorax_dept, on='CODE_DEPT')
gdf_idf_thorax_150m['pourcentage']= (gdf_idf_thorax_150m['nbre_patients_<=150m']/gdf_idf_thorax_150m['nb_patientes_thorax']*100).round(1)


gdf_idf_gyneco_150m = gpd.sjoin(gdf_idf_gyneco_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_gyneco_150m_dept = gdf_idf_gyneco_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_gyneco_150m = gdf_idf_gyneco_150m_dept.merge(gdf_idf_gyneco_dept, on='CODE_DEPT')
gdf_idf_gyneco_150m['pourcentage'] = (gdf_idf_gyneco_150m['nbre_patients_<=150m']/gdf_idf_gyneco_150m['nb_patientes_gyneco']*100).round(1)

gdf_idf_gyneco_150m

,CODE_DEPT,nbre_patients_<=150m,nb_patientes_gyneco,NOM_DEPT,pourcentage
0,75,5,373,PARIS,1.3
1,77,2,121,SEINE-ET-MARNE,1.7
2,78,2,352,YVELINES,0.6
3,91,1,120,ESSONNE,0.8
4,92,14,518,HAUTS-DE-SEINE,2.7
5,93,8,214,SEINE-SAINT-DENIS,3.7
6,94,7,107,VAL-DE-MARNE,6.5
7,95,1,134,VAL-D'OISE,0.7


In [801]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

fig.add_trace(go.Bar(x=gdf_idf_hemato_150m['NOM_DEPT'], y=gdf_idf_hemato_150m['pourcentage'], name='cancer hemato',
                     text=gdf_idf_hemato_150m['pourcentage'].round(0),
                     textposition='outside',textfont=dict(size=16)))
fig.add_trace(go.Bar(x=gdf_idf_sein_150m['NOM_DEPT'], y=gdf_idf_sein_150m['pourcentage'], name='cancer sein',
                     text=gdf_idf_sein_150m['pourcentage'].round(0),
                     textposition='outside',textfont=dict(size=16)))
fig.add_trace(go.Bar(x=gdf_idf_thorax_150m['NOM_DEPT'], y=gdf_idf_thorax_150m['pourcentage'], name='cancer thorax',
                     text=gdf_idf_thorax_150m['pourcentage'].round(0),
                     textposition='outside',textfont=dict(size=16)))
fig.add_trace(go.Bar(x=gdf_idf_gyneco_150m['NOM_DEPT'], y=gdf_idf_gyneco_150m['pourcentage'], name='cancer Gynéco',
                     text=gdf_idf_gyneco_150m['pourcentage'].round(0),
                     textposition='outside',textfont=dict(size=16)))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Département'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle repartie par département et située à une distance maximale de 150m des voies routières principales en fonction de leur groupe pathologique')

config = {
    'toImageButtonOptions': {
        'format': 'png',  # one of png, svg, jpeg, webp
        'filename': 'bar_stats_routes_150m',
        'height': 500,
        'width': 1500,
        'scale': 5  # Multiply title/legend/axis/canvas sizes by this factor
    }
}

fig.show(config=config)

### DANS UN PERIMETRE DE 500M 

In [802]:
pourcentage_500m = ((len(gdf_F_4patho_500m.drop_duplicates(subset = ['pseudo_provisoire'])))/len(gdf_patientes_idf.drop_duplicates(subset=['pseudo_provisoire']))*100)
print(f"{pourcentage_500m:.2f} % de patientes sont situées à moins de 500m des voies routières principales en IDF")

13.51 % de patientes sont situées à moins de 500m des voies routières principales en IDF


In [803]:
patientes_counts_exp_by_dept = gdf_F_4patho_500m[["CODE_DEPT"]].groupby('CODE_DEPT').size().reset_index(name = 'exposes')
patientes_counts_exp_by_dept
stats_500m = patientes_counts_by_dept.merge(patientes_counts_exp_by_dept, left_on='CODE_DEPT', right_on='CODE_DEPT')
stats_500m.sort_values(by='CODE_DEPT')
stats_500m["Pourcentage_500"] = (stats_500m["exposes"]/stats_500m["total"]*100).round(2)
stats_500m.sort_values(by='CODE_DEPT')


,CODE_DEPT,NOM_DEPT,total,exposes,Pourcentage_500
3,75,PARIS,7508,929,12.37
5,77,SEINE-ET-MARNE,2300,205,8.91
1,78,YVELINES,6383,618,9.68
6,91,ESSONNE,2168,194,8.95
2,92,HAUTS-DE-SEINE,7153,1494,20.89
0,93,SEINE-SAINT-DENIS,2410,562,23.32
4,94,VAL-DE-MARNE,2209,584,26.44
7,95,VAL-D'OISE,2082,252,12.10


In [804]:
fig = px.bar(stats_500m, x='NOM_DEPT', y='Pourcentage_500', title='Patientèle par departement située à une distance maximale de 500m des voies routieres principales',
             labels={'NOM_DEPT': 'Département', 'Pourcentage_500': 'patients(%)'})
fig.show()

In [805]:
gdf_idf_hemato_500m = gpd.sjoin(gdf_idf_hemato_points, gdf_buffer_500m, how='inner', predicate='intersects')
gdf_idf_hemato_500m_dept = gdf_idf_hemato_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_hemato_500m = gdf_idf_hemato_500m_dept.merge(gdf_idf_hemato_dept, on='CODE_DEPT')
gdf_idf_hemato_500m['pourcentage']= (gdf_idf_hemato_500m['nbre_patients_<=500m']/gdf_idf_hemato_500m['nb_patientes_hemato']*100).round(0)


gdf_idf_sein_500m = gpd.sjoin(gdf_idf_sein_points, gdf_buffer_500m, how='inner', predicate='intersects')
gdf_idf_sein_500m_dept = gdf_idf_sein_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_sein_500m = gdf_idf_sein_500m_dept.merge(gdf_idf_sein_dept, on='CODE_DEPT')
gdf_idf_sein_500m['pourcentage']= (gdf_idf_sein_500m['nbre_patients_<=500m']/gdf_idf_sein_500m['nb_patientes_sein']*100).round(0)


gdf_idf_thorax_500m = gpd.sjoin(gdf_idf_thorax_points, gdf_buffer_500m, how='inner', predicate='intersects')
gdf_idf_thorax_500m_dept = gdf_idf_thorax_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_thorax_500m = gdf_idf_thorax_500m_dept.merge(gdf_idf_thorax_dept, on='CODE_DEPT')
gdf_idf_thorax_500m['pourcentage']= (gdf_idf_thorax_500m['nbre_patients_<=500m']/gdf_idf_thorax_500m['nb_patientes_thorax']*100).round(0)


gdf_idf_gyneco_500m = gpd.sjoin(gdf_idf_gyneco_points, gdf_buffer_500m, how='inner', predicate='intersects')
gdf_idf_gyneco_500m_dept = gdf_idf_gyneco_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_gyneco_500m = gdf_idf_gyneco_500m_dept.merge(gdf_idf_gyneco_dept, on='CODE_DEPT')
gdf_idf_gyneco_500m['pourcentage'] = (gdf_idf_gyneco_500m['nbre_patients_<=500m']/gdf_idf_gyneco_150m['nb_patientes_gyneco']*100).round(0)

gdf_idf_gyneco_500m


,CODE_DEPT,nbre_patients_<=500m,nb_patientes_gyneco,NOM_DEPT,pourcentage
0,75,61,373,PARIS,16.0
1,77,9,121,SEINE-ET-MARNE,7.0
2,78,38,352,YVELINES,11.0
3,91,5,120,ESSONNE,4.0
4,92,124,518,HAUTS-DE-SEINE,24.0
5,93,65,214,SEINE-SAINT-DENIS,30.0
6,94,24,107,VAL-DE-MARNE,22.0
7,95,11,134,VAL-D'OISE,8.0


In [806]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_hemato_500m['NOM_DEPT'], y=gdf_idf_hemato_500m['pourcentage'], name='cancer hemato',
                     text=gdf_idf_hemato_500m['pourcentage'],
                     textposition='outside',textfont=dict(size=16)))
fig.add_trace(go.Bar(x=gdf_idf_sein_500m['NOM_DEPT'], y=gdf_idf_sein_500m['pourcentage'], name='cancer sein',
                     text=gdf_idf_sein_500m['pourcentage'],
                     textposition='outside',textfont=dict(size=16)))
fig.add_trace(go.Bar(x=gdf_idf_thorax_500m['NOM_DEPT'], y=gdf_idf_thorax_500m['pourcentage'], name='cancer thorax',
                     text=gdf_idf_thorax_500m['pourcentage'],
                     textposition='outside',textfont=dict(size=16)))
fig.add_trace(go.Bar(x=gdf_idf_gyneco_500m['NOM_DEPT'], y=gdf_idf_gyneco_500m['pourcentage'], name='cancer gyneco',
                     text=gdf_idf_gyneco_500m['pourcentage'],
                     textposition='outside',textfont=dict(size=16)))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Département'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle située à une distance maximale de 500m des voies routières principales par département et en fonction de leur pathologie')


config = {
    'toImageButtonOptions': {
        'format': 'png',  # one of png, svg, jpeg, webp
        'filename': 'bar_stats_routes_500m',
        'height': 500,
        'width': 1500,
        'scale': 5  # Multiply title/legend/axis/canvas sizes by this factor
    }
}

fig.show(config=config)

### DANS UN PERIMETRE DE 1KM 

In [807]:
pourcentage_1km = ((len(gdf_F_4patho_1km.drop_duplicates(subset = ['pseudo_provisoire'])))/len(gdf_patientes_idf.drop_duplicates(subset=['pseudo_provisoire']))*100)
print(f"{pourcentage_1km:.2f} % de patientes sont situées à moins d\'un kilometre des voies routières principales en IDF")

31.84 % de patientes sont situées à moins d'un kilometre des voies routières principales en IDF


In [808]:
patientes_counts_exp_by_dept = gdf_F_4patho_1km[["CODE_DEPT"]].groupby('CODE_DEPT').size().reset_index(name = 'exposes')
patientes_counts_exp_by_dept
stats_1km = patientes_counts_by_dept.merge(patientes_counts_exp_by_dept, left_on='CODE_DEPT', right_on='CODE_DEPT')
stats_1km.sort_values(by='CODE_DEPT')
stats_1km["Pourcentage_1km"] = (stats_1km["exposes"]/stats_1km["total"]*100).round(2)
stats_1km.sort_values(by='CODE_DEPT')


,CODE_DEPT,NOM_DEPT,total,exposes,Pourcentage_1km
3,75,PARIS,7508,2580,34.36
5,77,SEINE-ET-MARNE,2300,475,20.65
1,78,YVELINES,6383,1638,25.66
6,91,ESSONNE,2168,476,21.96
2,92,HAUTS-DE-SEINE,7153,3347,46.79
0,93,SEINE-SAINT-DENIS,2410,1132,46.97
4,94,VAL-DE-MARNE,2209,1088,49.25
7,95,VAL-D'OISE,2082,670,32.18


In [809]:
fig = px.bar(stats_1km, x='NOM_DEPT', y='Pourcentage_1km', title='Patientèle par departement située à une distance maximale d\'un km des voies routières principales',
             labels={'NOM_DEPT': 'Département', 'Pourcentage_1km': 'patients(%)'})
fig.show()

In [810]:
gdf_idf_hemato_1km = gpd.sjoin(gdf_idf_hemato_points, gdf_buffer_1km, how='inner', predicate='intersects')
gdf_idf_hemato_1km_dept = gdf_idf_hemato_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_hemato_1km = gdf_idf_hemato_1km_dept.merge(gdf_idf_hemato_dept, on='CODE_DEPT')
gdf_idf_hemato_1km['pourcentage']= (gdf_idf_hemato_1km['nbre_patients_<=1km']/gdf_idf_hemato_1km['nb_patientes_hemato']*100).round(0)


gdf_idf_sein_1km = gpd.sjoin(gdf_idf_sein_points, gdf_buffer_1km, how='inner', predicate='intersects')
gdf_idf_sein_1km_dept = gdf_idf_sein_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_sein_1km = gdf_idf_sein_1km_dept.merge(gdf_idf_sein_dept, on='CODE_DEPT')
gdf_idf_sein_1km['pourcentage']= (gdf_idf_sein_1km['nbre_patients_<=1km']/gdf_idf_sein_1km['nb_patientes_sein']*100).round(0)


gdf_idf_thorax_1km = gpd.sjoin(gdf_idf_thorax_points, gdf_buffer_1km, how='inner', predicate='intersects')
gdf_idf_thorax_1km_dept = gdf_idf_thorax_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_thorax_1km = gdf_idf_thorax_1km_dept.merge(gdf_idf_thorax_dept, on='CODE_DEPT')
gdf_idf_thorax_1km['pourcentage']= (gdf_idf_thorax_1km['nbre_patients_<=1km']/gdf_idf_thorax_1km['nb_patientes_thorax']*100).round(0)


gdf_idf_gyneco_1km = gpd.sjoin(gdf_idf_gyneco_points, gdf_buffer_1km, how='inner', predicate='intersects')
gdf_idf_gyneco_1km_dept = gdf_idf_gyneco_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_gyneco_1km = gdf_idf_gyneco_1km_dept.merge(gdf_idf_gyneco_dept, on='CODE_DEPT')
gdf_idf_gyneco_1km['pourcentage'] = (gdf_idf_gyneco_1km['nbre_patients_<=1km']/gdf_idf_gyneco_1km['nb_patientes_gyneco']*100).round(0)

gdf_idf_gyneco_1km

,CODE_DEPT,nbre_patients_<=1km,nb_patientes_gyneco,NOM_DEPT,pourcentage
0,75,140,373,PARIS,38.0
1,77,24,121,SEINE-ET-MARNE,20.0
2,78,97,352,YVELINES,28.0
3,91,22,120,ESSONNE,18.0
4,92,253,518,HAUTS-DE-SEINE,49.0
5,93,123,214,SEINE-SAINT-DENIS,57.0
6,94,50,107,VAL-DE-MARNE,47.0
7,95,40,134,VAL-D'OISE,30.0


In [811]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_hemato_1km['NOM_DEPT'], y=gdf_idf_hemato_1km['pourcentage'], name='cancer hemato',
                     text=gdf_idf_hemato_1km['pourcentage'],
                     textposition='outside',textfont=dict(size=16)))
fig.add_trace(go.Bar(x=gdf_idf_sein_1km['NOM_DEPT'], y=gdf_idf_sein_1km['pourcentage'], name='cancer sein',
                     text=gdf_idf_sein_1km['pourcentage'],
                     textposition='outside',textfont=dict(size=16)))
fig.add_trace(go.Bar(x=gdf_idf_thorax_1km['NOM_DEPT'], y=gdf_idf_thorax_1km['pourcentage'], name='cancer thorax',
                     text=gdf_idf_thorax_1km['pourcentage'],
                     textposition='outside',textfont=dict(size=16)))
fig.add_trace(go.Bar(x=gdf_idf_gyneco_1km['NOM_DEPT'], y=gdf_idf_gyneco_1km['pourcentage'], name='cancer gyneco',
                     text=gdf_idf_gyneco_1km['pourcentage'],
                     textposition='outside',textfont=dict(size=16)))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Département'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle située à une distance maximale d\'un km des voies routières principales par département et en fonction de leur pathologie')

config = {
    'toImageButtonOptions': {
        'format': 'png',  # one of png, svg, jpeg, webp
        'filename': 'bar_stats_routes_1km',
        'height': 500,
        'width': 1500,
        'scale': 5  # Multiply title/legend/axis/canvas sizes by this factor
    }
}

fig.show(config=config)

# ANALYSE SOCIO ECONOMIQUE

In [812]:
df_F_rev_chom = df_patients_revenus_chomage[df_patients_revenus_chomage["patient_sexe"]=="F"]
df_F_IDF_rev_chom = df_F_rev_chom[df_F_rev_chom['INSEE_REG']==11]
df_F_Hors_IDF_rev_chom = df_F_rev_chom[df_F_rev_chom['INSEE_REG']!=11]

##### Bords revenus et chomages

In [813]:
df_revenus = pd.read_csv("data/insee_revenu/BASE_TD_FILO_DISP_IRIS_2020.csv",sep=";",dtype=str)[["IRIS","DISP_MED20"]]
df_revenus[df_revenus["DISP_MED20"]=="ns"] = np.nan
df_revenus[df_revenus["DISP_MED20"]=="nd"] = np.nan
df_revenus["DISP_MED20"] = df_revenus["DISP_MED20"].astype(float)
quartiles, bords_revenus = pd.qcut(df_revenus["DISP_MED20"], 4, labels=False,retbins=True)


df_chomage = pd.read_csv("data/socio_eco/chomage/base-ic-activite-residents-2020.CSV",sep=";",dtype={'IRIS':str,'P20_ACT1564':float,'P20_CHOM1564':float})[["IRIS","P20_ACT1564","P20_CHOM1564"]]
df_chomage["P20_taux_CHOM1564"]=df_chomage["P20_CHOM1564"]/df_chomage["P20_ACT1564"]*100 
quartile, bords_chomage = pd.qcut(df_chomage["P20_taux_CHOM1564"], 4, labels=False,retbins=True)

C:\Users\lpokambo\AppData\Local\Temp\ipykernel_20512\1805345138.py:8: DtypeWarning:

Columns (1,3) have mixed types. Specify dtype option on import or set low_memory=False.



##### 1-En france

In [814]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0,4,1)]
quartile_count = df_revenus.value_counts().sort_index()
colors = ['#cce5ff', '#99ccff', '#66b2ff', '#0073e6']


fig = go.Figure(
    data=[go.Pie(
        labels=labels,
        values=quartile_count.values,
        sort=False,
        hole=.3
        )
    ])
fig.update_layout(title="Répartition en quartiles des revenus disponibles médians associé au domicile de notre patientèle française (en €)")
fig.update_traces(textposition='inside', textinfo='percent',marker=dict(colors=colors))
fig.show()

In [815]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0,4,1)]
quartile_count = df_F_rev_chom['quartile_revenu'].value_counts().sort_index()
colors = ['#cce5ff', '#99ccff', '#66b2ff', '#0073e6']


fig = go.Figure(
    data=[go.Pie(
        labels=labels,
        values=quartile_count.values,
        sort=False,
        hole=.3
        )
    ])
fig.update_layout(title="Répartition en quartiles des revenus disponibles médians associé au domicile de notre patientèle française (en €)")
fig.update_traces(textposition='inside', textinfo='percent',marker=dict(colors=colors))
fig.show()

In [816]:
labels = [f"Q{i+1}: {np.round(bords_chomage[i],2)} % - {np.round(bords_chomage[i+1],2)} %" for i in range(0,len(bords_chomage)-1,1)]
quartile_count = df_F_rev_chom['quartile_chomage'].value_counts().sort_index()
colors = ['#0073e6','#66b2ff','#99ccff','#cce5ff']

fig = go.Figure(
    data=[go.Pie(
        labels=labels,
        values=quartile_count.values,
        sort=False,
        hole=.3
        )
    ])
fig.update_layout(title="Répartition en quartiles du taux de chomage associé au domicile de notre patientèle française (en %)")
fig.update_traces(textposition='inside', textinfo='percent',marker=dict(colors=colors))
fig.show()

##### En Ile de France

In [817]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0,4,1)]
quartile_count = df_F_IDF_rev_chom['quartile_revenu'].value_counts().sort_index()
colors = ['#cce5ff', '#99ccff', '#66b2ff', '#0073e6']


fig = go.Figure(
    data=[go.Pie(
        labels=labels,
        values=quartile_count.values,
        sort=False,
        hole=.3
        )
    ])
fig.update_layout(title="Répartition en quartiles des revenus disponibles médians associé au domicile des patientes situées en IDF (en €)")
fig.update_traces(textposition='inside', textinfo='percent',marker=dict(colors=colors))
fig.show()

In [818]:
labels = [f"Q{i+1}: {np.round(bords_chomage[i],2)} % - {np.round(bords_chomage[i+1],2)} %" for i in range(0,len(bords_chomage)-1,1)]
quartile_count = df_F_Hors_IDF_rev_chom['quartile_chomage'].value_counts().sort_index()
colors = ['#0073e6','#66b2ff','#99ccff','#cce5ff']

fig = go.Figure(
    data=[go.Pie(
        labels=labels,
        values=quartile_count.values,
        sort=False,
        hole=.3
        )
    ])
fig.update_layout(title="Répartition en quartiles du taux de chomage associé au domicile de des patientes situées en IDF (en %)")
fig.update_traces(textposition='inside', textinfo='percent',marker=dict(colors=colors))
fig.show()

#### Hors d'ile de France

In [819]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0,4,1)]
quartile_count = df_F_Hors_IDF_rev_chom['quartile_revenu'].value_counts().sort_index()
colors = ['#cce5ff', '#99ccff', '#66b2ff', '#0073e6']


fig = go.Figure(
    data=[go.Pie(
        labels=labels,
        values=quartile_count.values,
        sort=False,
        hole=.3
        )
    ])
fig.update_layout(title="Répartition en quartiles des revenus disponibles médians associé au domicile des patientes situées Hors d'Ile de France (en €)")
fig.update_traces(textposition='inside', textinfo='percent',marker=dict(colors=colors))
fig.show()

In [820]:
labels = [f"Q{i+1}: {np.round(bords_chomage[i],2)} % - {np.round(bords_chomage[i+1],2)} %" for i in range(0,len(bords_chomage)-1,1)]
quartile_count = df_F_Hors_IDF_rev_chom['quartile_chomage'].value_counts().sort_index()
colors = ['#0073e6','#66b2ff','#99ccff','#cce5ff']

fig = go.Figure(
    data=[go.Pie(
        labels=labels,
        values=quartile_count.values,
        sort=False,
        hole=.3
        )
    ])
fig.update_layout(title="Répartition en quartiles du taux de chomage associé au domicile de des patientes situées hors d' Ile de France (en %)")
fig.update_traces(textposition='inside', textinfo='percent',marker=dict(colors=colors))
fig.show()

In [821]:
labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0, 4, 1)]
colors = ['#cce5ff', '#99ccff', '#66b2ff', '#0073e6']

quartile_count_fr = df_F_rev_chom['quartile_revenu'].value_counts().sort_index()
quartile_count_idf = df_F_IDF_rev_chom['quartile_revenu'].value_counts().sort_index()
quartile_count_hors_idf = values=df_F_Hors_IDF_rev_chom['quartile_revenu'].value_counts().sort_index()
# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text=f"Répartition en quartiles des revenus médians associé a notre patientèle atteinte de cancer",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent',marker=dict(colors=colors))

# Show the figure
fig.show()

In [822]:
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])
labels = [f"Q{i+1}: {np.round(bords_chomage[i],2)} % - {np.round(bords_chomage[i+1],2)} %" for i in range(0,len(bords_chomage)-1,1)]
colors = ['#0073e6','#66b2ff','#99ccff','#cce5ff']
quartile_count_fr = df_F_rev_chom['quartile_chomage'].value_counts().sort_index()
quartile_count_idf = df_F_IDF_rev_chom['quartile_chomage'].value_counts().sort_index()
quartile_count_hors_idf = df_F_Hors_IDF_rev_chom['quartile_chomage'].value_counts().sort_index()

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

# Add pie charts
fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

# Update layout
fig.update_layout(title_text=f"Répartition en quartiles du taux de chomage associé au domicile <br> de notre patientèle atteinte de cancer (en %)",
                  showlegend=True)

# Update traces
fig.update_traces(textposition='inside', textinfo='percent',marker=dict(colors=colors))

# Show the figure
fig.show()

#### Socio-economique par pathologie

In [823]:
pathologie = ["Sein", "Gynéco", "Thorax", "Hemato"]

In [824]:

# for patho in pathologie:
#     patientes_patho = df_F_rev_chom[df_F_rev_chom["patho"]==patho]
#     patientes_patho_idf = patientes_patho[patientes_patho["INSEE_REG"]==11]
#     patientes_patho_idf_hors_idf = patientes_patho[patientes_patho["INSEE_REG"]!=11]

#     labels = [f"Q{i+1}: {np.round(bords_revenus[i],2)} € - {np.round(bords_revenus[i+1],2)} €" for i in range(0, 4, 1)]
#     colors = ['#cce5ff', '#99ccff', '#66b2ff', '#0073e6']

#     quartile_count_fr = patientes_patho['quartile_revenu'].value_counts().sort_index()
#     quartile_count_idf = patientes_patho_idf['quartile_revenu'].value_counts().sort_index()
#     quartile_count_hors_idf = patientes_patho_idf_hors_idf['quartile_revenu'].value_counts().sort_index()

#     # Create subplots
#     fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

#     # Add pie charts
#     fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
#     fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
#     fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

#     # Update layout
#     fig.update_layout(title_text=f"Répartition en quartiles des revenus  médians associé a notre patientèle atteinte de cancer {patho}",
#                   showlegend=True)

#     # Update traces
#     fig.update_traces(textposition='inside', textinfo='percent',marker=dict(colors=colors))

  
#     # Show the figure
#     config = {
#   'toImageButtonOptions': {
#     'format': 'png', # one of png, svg, jpeg, webp
#     'filename': f'revenus_patientes_{patho}',
#     'height': 1080,
#     'width': 1920,
#     'scale':6 # Multiply title/legend/axis/canvas sizes by this factor
#     }
# }

#     fig.show(config=config)
    
   
    
#     labels = [f"Q{i+1}: {np.round(bords_chomage[i],2)} % - {np.round(bords_chomage[i+1],2)} %" for i in range(0,len(bords_chomage)-1,1)]
#     colors = ['#0073e6','#66b2ff','#99ccff','#cce5ff']
#     quartile_count_fr = patientes_patho['quartile_chomage'].value_counts().sort_index()
#     quartile_count_idf = patientes_patho_idf['quartile_chomage'].value_counts().sort_index()
#     quartile_count_hors_idf = patientes_patho_idf_hors_idf['quartile_chomage'].value_counts().sort_index()

#     # Create subplots
#     fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

#     # Add pie charts
#     fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
#     fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
#     fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

#     # Update layout
#     fig.update_layout(title_text=f"Répartition en quartiles du taux de chomage associé au domicile <br> de notre patientèle atteinte de cancer {patho}(en %)",
#                   showlegend=True)

#     # Update traces
#     fig.update_traces(textposition='inside', textinfo='percent',marker=dict(colors=colors))

#     # Show the figure
#     config = {
#   'toImageButtonOptions': {
#     'format': 'png', # one of png, svg, jpeg, webp
#     'filename': f'chomage_patientes_{patho}',
#     'height': 1080,
#     'width': 700,
#     'scale':6 # Multiply title/legend/axis/canvas sizes by this factor
#   }
# }


#     fig.show(config=config)
    


In [825]:
for patho in pathologie:
    patientes_patho = df_F_rev_chom[df_F_rev_chom["patho"] == patho]
    patientes_patho_idf = patientes_patho[patientes_patho["INSEE_REG"] == 11]
    patientes_patho_idf_hors_idf = patientes_patho[patientes_patho["INSEE_REG"] != 11]

    labels = [f"Q{i+1}: {np.round(bords_revenus[i], 2)} € - {np.round(bords_revenus[i+1], 2)} €" for i in range(0, 4, 1)]
    colors = ['#cce5ff', '#99ccff', '#66b2ff', '#0073e6']

    quartile_count_fr = patientes_patho['quartile_revenu'].value_counts().sort_index()
    quartile_count_idf = patientes_patho_idf['quartile_revenu'].value_counts().sort_index()
    quartile_count_hors_idf = patientes_patho_idf_hors_idf['quartile_revenu'].value_counts().sort_index()

    # Create subplots
    fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

    # Add pie charts
    fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
    fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
    fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

    # Update layout
    fig.update_layout(
        title_text=f"Répartition en quartiles des revenus médians associé a notre patientèle atteinte de cancer {patho}",
        showlegend=True,   
    )

    # Update traces
    fig.update_traces(textposition='inside', textinfo='percent', textfont=dict(size=20), marker=dict(colors=colors))

    # Show the figure
    config = {
        'toImageButtonOptions': {
            'format': 'png',  # one of png, svg, jpeg, webp
            'filename': f'revenus_patientes_{patho}',
            'height': 1080,
            'width': 1920,
            'scale': 6  # Multiply title/legend/axis/canvas sizes by this factor
        }
    }

    fig.show(config=config)

    labels = [f"Q{i+1}: {np.round(bords_chomage[i], 2)} % - {np.round(bords_chomage[i+1], 2)} %" for i in range(0, len(bords_chomage)-1, 1)]
    colors = ['#0073e6', '#66b2ff', '#99ccff', '#cce5ff']
    quartile_count_fr = patientes_patho['quartile_chomage'].value_counts().sort_index()
    quartile_count_idf = patientes_patho_idf['quartile_chomage'].value_counts().sort_index()
    quartile_count_hors_idf = patientes_patho_idf_hors_idf['quartile_chomage'].value_counts().sort_index()

    # Create subplots
    fig = make_subplots(rows=1, cols=3, subplot_titles=("France", "IDF", "Hors IDF"), specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]])

    # Add pie charts
    fig.add_trace(go.Pie(labels=labels, values=quartile_count_fr.values, sort=False, hole=.3, name="France"), row=1, col=1)
    fig.add_trace(go.Pie(labels=labels, values=quartile_count_idf.values, sort=False, hole=.3, name="IDF"), row=1, col=2)
    fig.add_trace(go.Pie(labels=labels, values=quartile_count_hors_idf.values, sort=False, hole=.3, name="Hors IDF"), row=1, col=3)

    # Update layout
    fig.update_layout(
        title_text=f"Répartition en quartiles du taux de chomage associé au domicile <br> de notre patientèle atteinte de cancer {patho}(en %)",
        showlegend=True,
    )

    # Update traces
    fig.update_traces(textposition='inside', textinfo='percent', textfont=dict(size=20), marker=dict(colors=colors))

    # Show the figure
    config = {
        'toImageButtonOptions': {
            'format': 'png',  # one of png, svg, jpeg, webp
            'filename': f'chomage_patientes_{patho}',
            'height': 1080,
            'width': 1920,
            'scale': 6  # Multiply title/legend/axis/canvas sizes by this factor
        }
    }

    fig.show(config=config)